# Notebook 07 — Colab E2E CV Launch
## SentinelFatal2 — End-to-End 5-Fold Cross-Validation (Overnight Run)

**Run this notebook top-to-bottom in Google Colab with a T4 GPU.**

| Cell | Action | Time |
|------|--------|------|
| 1 | **GPU CHECK** — confirm T4 before proceeding | seconds |
| 2 | Mount Drive + clone repo | ~1 min |
| 3 | Data setup (download from GitHub if needed) | ~1–3 min |
| 4 | Install dependencies | ~1 min |
| 5 | **Dry-run** — verify full pipeline works on GPU | ~5 min |
| 6 | **Overnight run** — launch in background via nohup | 3–5 hours |
| 7 | Morning check — read results | seconds |

> **STOP after Cell 1** and confirm you have a GPU before continuing.

In [ ]:
# ── Cell 1: GPU CHECK ── RUN THIS FIRST AND CONFIRM BEFORE CONTINUING ────────
import subprocess, sys

# nvidia-smi summary
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)

import torch
has_gpu = torch.cuda.is_available()

print("=" * 60)
print("  SentinelFatal2 — Hardware Check")
print("=" * 60)
if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU : {gpu_name}")
    print(f"  VRAM: {gpu_mem:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}")
    print()
    if result.returncode == 0:
        print("  nvidia-smi:")
        for line in result.stdout.strip().split('\n'):
            print(f"    {line}")
    DEVICE = 'cuda'
    print()
    print("  ✅ GPU is available — safe to continue.")
else:
    print("  ⚠️  NO GPU detected.")
    print("  PyTorch sees only CPU.")
    print()
    print("  To fix in Colab:")
    print("    Runtime → Change runtime type → T4 GPU → Save")
    print("    Then: Runtime → Run all (or re-run from Cell 1)")
    print()
    print("  If you continue on CPU, the full run will take ~25 hours.")
    DEVICE = 'cpu'

print("=" * 60)
print(f"  DEVICE = '{DEVICE}'")
print("=" * 60)

# STOP HERE — confirm GPU before running Cell 2+
if not has_gpu:
    raise RuntimeError("No GPU found. Change runtime type to T4 GPU and restart.")

In [ ]:
# ── Cell 2: Clone / Update Repository ────────────────────────────────────────
import os
from pathlib import Path

REPO_URL  = "https://github.com/ArielShamay/SentinelFatal2.git"
REPO_DIR  = Path("/content/SentinelFatal2")

if REPO_DIR.exists():
    print("Repository already exists — pulling latest changes...")
    os.chdir(REPO_DIR)
    result = os.popen("git pull").read()
    print(result)
else:
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

print(f"\nWorking directory: {os.getcwd()}")

# Add repo root to Python path so src/ is importable
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("✅ Repository ready.")

In [ ]:
# ── Cell 3: Data Setup — Download & Extract from GitHub ──────────────────────
# The processed .npy files (~100 MB) live in data_processed.zip on GitHub.
# We download it once; Colab's /content persists within the same session.

import os, zipfile, time
from pathlib import Path

REPO_DIR      = Path("/content/SentinelFatal2")
PROCESSED_DIR = REPO_DIR / "data" / "processed" / "ctu_uhb"
ZIP_PATH      = REPO_DIR / "data_processed.zip"
ZIP_URL       = "https://raw.githubusercontent.com/ArielShamay/SentinelFatal2/master/data_processed.zip"

# ── Check existing .npy files ────────────────────────────────────────────────
ctu_npy   = list(PROCESSED_DIR.glob("*.npy")) if PROCESSED_DIR.exists() else []
fhrma_dir = REPO_DIR / "data" / "processed" / "fhrma"
fhrma_npy = list(fhrma_dir.glob("*.npy")) if fhrma_dir.exists() else []

print(f"Existing .npy files: ctu_uhb={len(ctu_npy)}, fhrma={len(fhrma_npy)}")

if len(ctu_npy) >= 552 and len(fhrma_npy) >= 135:
    print("✅ Processed data already present — skipping download.")
else:
    print(f"\nDownloading {ZIP_URL} ...")
    t0 = time.time()
    os.system(f"wget -q --show-progress -O {ZIP_PATH} '{ZIP_URL}'")
    print(f"   Downloaded in {time.time()-t0:.0f}s")

    print("\nExtracting ZIP ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f"   Extracted to {REPO_DIR}")
    os.remove(ZIP_PATH)  # free space

# ── Final verification ───────────────────────────────────────────────────────
ctu_npy   = list(PROCESSED_DIR.glob("*.npy"))
fhrma_npy = list((REPO_DIR / "data" / "processed" / "fhrma").glob("*.npy")) if fhrma_dir.exists() else []

print(f"\nVerification:")
print(f"  ctu_uhb .npy : {len(ctu_npy)} (expected 552–553)")
print(f"  fhrma .npy   : {len(fhrma_npy)} (expected 135–136)")

# Check split CSVs
for split in ['train', 'val', 'test']:
    p = REPO_DIR / "data" / "splits" / f"{split}.csv"
    print(f"  {split}.csv  : {'✅' if p.exists() else '❌ MISSING'}")

# Check pretrain checkpoint
pt_ckpt = REPO_DIR / "checkpoints" / "pretrain" / "best_pretrain.pt"
ft_ckpt = REPO_DIR / "checkpoints" / "finetune" / "best_finetune.pt"
print(f"  best_pretrain.pt : {'✅' if pt_ckpt.exists() else '❌ MISSING'}")
print(f"  best_finetune.pt : {'✅' if ft_ckpt.exists() else '❌ MISSING'}")

if len(ctu_npy) < 552:
    raise RuntimeError(f"Expected 552+ ctu_uhb .npy files, found {len(ctu_npy)}. Download may have failed.")

In [ ]:
# ── Cell 4: Install Dependencies ─────────────────────────────────────────────
# Colab already has numpy, pandas, matplotlib, scipy, scikit-learn, torch.
# We only need to confirm PyYAML and tqdm are available.

import importlib, subprocess, sys

REQUIRED = {
    "yaml":         "PyYAML>=6.0",
    "tqdm":         "tqdm>=4.65",
    "sklearn":      "scikit-learn>=1.3",
}

to_install = []
for module, pkg in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        to_install.append(pkg)

if to_install:
    print(f"Installing: {to_install}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
else:
    print("✅ All dependencies already installed.")

# Quick sanity imports
import torch, numpy as np, pandas as pd, yaml, sklearn
print(f"\nPackage versions:")
print(f"  torch      : {torch.__version__}  (CUDA: {torch.version.cuda})")
print(f"  numpy      : {np.__version__}")
print(f"  pandas     : {pd.__version__}")
print(f"  sklearn    : {sklearn.__version__}")
print(f"  PyYAML     : {yaml.__version__}")
print(f"\n  Device     : {DEVICE}")

In [ ]:
# ── Cell 5: DRY-RUN — Verify full pipeline works on GPU (~5 min) ─────────────
# Runs 1 fold with max_batches=2. Should complete without errors.
# If this cell fails, DO NOT proceed to the overnight run.

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
os.chdir(REPO_DIR)

print("Starting dry-run (1 fold, max_batches=2)...")
print("Estimated time: 3–6 minutes on T4 GPU\n")

cmd = [
    sys.executable, "scripts/run_e2e_cv.py",
    "--device", DEVICE,
    "--dry-run",
    "--folds", "1",
    "--force-folds",
    "--seed", "42",
]

result = subprocess.run(cmd, capture_output=False, text=True)

print("\n" + "=" * 60)
if result.returncode == 0:
    print("✅ Dry-run PASSED — all phases completed successfully.")
    print("   Safe to launch overnight run (Cell 6).")
else:
    print(f"❌ Dry-run FAILED (exit code {result.returncode}).")
    print("   Check output above. DO NOT run Cell 6 until this passes.")
print("=" * 60)

In [ ]:
# ── Cell 6: OVERNIGHT RUN — Launch Full 5-Fold E2E CV in Background ──────────
# Uses nohup so the process keeps running even if the browser tab becomes idle.
# The script auto-resumes if interrupted: completed folds are skipped.
#
# AFTER launching:
#   1. Run the next cell once to confirm it started
#   2. You can close the browser tab — the process continues on Colab's server
#   3. Re-open in the morning and run Cell 7 to check results

import os, subprocess
from pathlib import Path

REPO_DIR  = Path("/content/SentinelFatal2")
LOG_FILE  = REPO_DIR / "logs" / "e2e_cv_master.log"
(REPO_DIR / "logs").mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

cmd = (
    f"nohup python scripts/run_e2e_cv.py "
    f"--device {DEVICE} "
    f"--force-folds "
    f"--folds 5 "
    f"--stride 60 "
    f"--seed 42 "
    f"> {LOG_FILE} 2>&1 &"
)

print("Launching overnight run...")
print(f"Command: {cmd}\n")
os.system(cmd)

import time
time.sleep(5)  # give the process a moment to start

# Read PID from nohup
pid_result = subprocess.run(["pgrep", "-f", "run_e2e_cv.py"], capture_output=True, text=True)
pid = pid_result.stdout.strip()

print(f"Process PID: {pid if pid else '(check below)'}")
print(f"Log file:    {LOG_FILE}")
print()
print("First 30 lines of log:")
print("-" * 60)
os.system(f"head -30 {LOG_FILE}")

In [ ]:
# ── Cell 7: MORNING CHECK — Read Results ─────────────────────────────────────
# Run this cell in the morning to check on the run.
# Can be run safely at any time — it only reads files, never modifies anything.

import os, pandas as pd
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
LOG_FILE = REPO_DIR / "logs" / "e2e_cv_master.log"
PROGRESS = REPO_DIR / "logs" / "e2e_cv_progress.csv"
REPORT   = REPO_DIR / "results" / "e2e_cv_final_report.csv"

print("=" * 60)
print("  SentinelFatal2 — E2E CV Morning Check")
print("=" * 60)

# Is the process still running?
pid_check = os.popen("pgrep -f run_e2e_cv.py").read().strip()
if pid_check:
    print(f"\n  Process still running (PID: {pid_check})")
else:
    print("\n  Process has finished (or was interrupted).")

# Last 40 lines of log
print("\n-- Last 40 lines of log " + "-" * 36)
os.system(f"tail -40 {LOG_FILE}")

# Per-fold progress
print("\n-- Per-fold progress " + "-" * 39)
if PROGRESS.exists():
    df_prog = pd.read_csv(PROGRESS)
    print(df_prog.to_string(index=False))
else:
    print("  (no progress file yet — run may still be in fold 0)")

# Final report
print("\n-- Final report " + "-" * 44)
if REPORT.exists():
    df_rep = pd.read_csv(REPORT)
    print(df_rep.to_string(index=False))
    print("\n✅ Run COMPLETE.")
else:
    print("  (no final report yet — run has not completed all 5 folds)")

print("\n-- ROC curve plot " + "-" * 42)
img_path = REPO_DIR / "docs" / "images" / "e2e_cv.png"
if img_path.exists():
    from IPython.display import Image, display
    display(Image(str(img_path)))
else:
    print("  (plot not yet generated)")
print("=" * 60)